## 2.3 无编码 BPSK BER 扫描

在上一节中，我们学习了信道编码的基本思想和 Polar 码的编解码原理。在引入编码之前，需要先建立无编码 BPSK 的 BER 基线，作为后续编码对比的参照。

本节首先拆解 BPSK 的调制解调每一步并画图展示，然后扫描多个 SNR 统计 BER，与理论值对比验证。

本节学习大纲如下：

- BPSK 调制解调具体步骤
- BPSK的 BER 扫描
- 与理论值对比并绘制曲线

### 本实验涉及的关键文件

```
src/nearlink_sdr/
├── phy/
│   ├── psk.py               <- PSKModulator: BPSK 调制 (星座映射+RRC)
│   │                           PSKDemodulator: 匹配滤波 + 判决
│   └── channel.py           <- ChannelModel: AWGN 信道模型
```


---

### 1. BPSK 调制解调步骤

**步骤 1：BPSK 星座映射** — 0 → +1, 1 → -1

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
tx_bits = rng.integers(0, 2, 100)

symbols = 1.0 - 2.0 * tx_bits.astype(float)

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.scatter(np.arange(20), symbols[:20], c="b", s=30, zorder=3)
ax.set_xlabel("Bit Index"); ax.set_ylabel("Amplitude")
ax.set_xticks(np.arange(20))
ax.set_title("Step 1: BPSK Mapping (0->+1, 1->-1)")
ax.set_ylim(-1.5, 1.5); ax.set_yticks([-1, 0, 1])
ax.grid(True, ls="--", alpha=0.3); plt.show()

**步骤 2：上采样** — 每符号插零

In [ ]:
sps = 4
upsampled = np.zeros(len(symbols) * sps, dtype=complex)
upsampled[::sps] = symbols

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.stem(np.arange(80), upsampled[:80].real, linefmt="b-", markerfmt="bo", basefmt="gray")
ax.set_xlabel("Sample Index"); ax.set_ylabel("Amplitude")
ax.set_title("Step 2: Upsampling (zero insertion)")
ax.set_ylim(-1.5, 1.5); ax.grid(True, ls="--", alpha=0.3)
plt.show()

**步骤 3：RRC 脉冲成型** — 卷积 RRC 滤波器，把冲激串平滑为连续波形

In [ ]:
from nearlink_sdr.phy.psk import PSKModulator
mod = PSKModulator(mod_type="BPSK", sps=sps)
tx_signal = np.convolve(upsampled, mod._rrc, mode="same")

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.plot(np.arange(80), np.real(tx_signal[:80]), "b-", lw=0.8)
ax.set_xlabel("Sample Index"); ax.set_ylabel("Amplitude")
ax.set_title("Step 3: After RRC Pulse Shaping")
ax.set_ylim(-0.8, 0.8); ax.grid(True, ls="--", alpha=0.3)
plt.show()

**步骤 4: 加入 AWGN 并进行匹配滤波**

In [ ]:
from nearlink_sdr.phy.channel import ChannelModel
ch = ChannelModel(snr_db=6.0)
rx_signal = ch.apply_awgn(tx_signal, sps)
matched = np.convolve(rx_signal, mod._rrc, mode="same")

fig, ax = plt.subplots(figsize=(10, 3))
t = np.arange(120)
ax.plot(t, np.real(rx_signal[:120]), "r-", alpha=0.4, lw=1, label="Before")
ax.plot(t, np.real(matched[:120]), "b-", lw=1, label="After matched filter")
ax.legend(fontsize=8); ax.set_xlabel("Sample Index"); ax.set_ylabel("Amplitude")
ax.set_title("Step 5: Matched Filtering"); ax.set_ylim(-1.5, 1.5)
ax.grid(True, ls="--", alpha=0.3); plt.show()
print("上图是匹配滤波前后的波形对比")

**步骤 5: 下采样** — 每 sps 个采样点取一个符号点

In [ ]:
sampled = matched[::sps][:len(symbols)]

fig, ax = plt.subplots(figsize=(10, 3))
t = np.arange(120)
ax.plot(t, np.real(matched[:120]), "b-", lw=0.6, alpha=0.5, label="Matched output")
t_sym = np.arange(30) * sps
ax.scatter(t_sym, np.real(sampled[:30]), c="red", s=30, zorder=5, label="Sampled points")
ax.legend(fontsize=8); ax.set_xlabel("Sample Index"); ax.set_ylabel("Amplitude")
ax.set_title("Step 6: Downsampling (every sps-th sample)"); ax.set_ylim(-1.5, 1.5)
ax.grid(True, ls="--", alpha=0.3); plt.show()

**步骤 7: 判决** — 符号值 > 0 判为 0, < 0 判为 1

In [ ]:
rx_bits_manual = (np.real(sampled) < 0).astype(int)
errors = int(np.sum(tx_bits[:len(rx_bits_manual)] != rx_bits_manual))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 3.5))
ax1.scatter(np.arange(20), tx_bits[:20], c="b", s=20, zorder=3, label="TX")
ax1.set_ylim(-0.1, 1.1); ax1.set_ylabel("TX bits"); ax1.legend(fontsize=8)
ax1.set_xlim(0, 19); ax1.set_xticks(np.arange(20))
ax1.grid(True, ls="--", alpha=0.3)
ax2.scatter(np.arange(20), rx_bits_manual[:20], c="r", s=20, zorder=3, label="RX")
ax2.set_ylim(-0.1, 1.1); ax2.set_xlabel("Bit Index"); ax2.set_ylabel("RX bits")
ax2.set_xlim(0, 19); ax2.set_xticks(np.arange(20))
ax2.legend(fontsize=8); ax2.grid(True, ls="--", alpha=0.3)
plt.suptitle(f"Step 7: Decision ({errors} errors / {len(rx_bits_manual)} bits)")
plt.tight_layout(); plt.show()

---

### 2. BER-SNR 扫描

In [ ]:
from nearlink_sdr.phy.psk import PSKDemodulator
num_bits = 5000
tx_bits_ber = rng.integers(0, 2, num_bits)   # 独立变量, 不影响步骤 1-7 的 tx_bits
mod = PSKModulator(mod_type="BPSK", sps=4)
demod = PSKDemodulator(mod_type="BPSK", sps=4)

snr_range = np.arange(0, 16, 2)
ber_list = []
for snr in snr_range:
    tx_signal = mod.modulate(tx_bits_ber)
    ch = ChannelModel(snr_db=float(snr))
    rx_signal = ch.apply_awgn(tx_signal, 4)
    rx_bits = demod.demodulate(rx_signal)
    n = min(len(tx_bits_ber), len(rx_bits))
    ber = np.mean(tx_bits_ber[:n] != rx_bits[:n])
    ber_list.append(ber)
    print(f"SNR={snr:2d} dB  BER={ber:.6f}")

### 3. 与理论值对比

无编码 BPSK 的理论 BER 公式为 BER = Q(sqrt(2Eb/N0))。

In [ ]:
from scipy.special import erfc

print(f"{'SNR':>5s}  {'SimBER':>10s}  {'TheoryBER':>10s}  {'Ratio':>6s}")
for snr, b in zip(snr_range, ber_list):
    snr_lin = 10**(snr/10)
    t = 0.5 * erfc(np.sqrt(snr_lin))
    r = b/t if t > 1e-10 else 0
    print(f"{snr:5.0f}  {b:10.6f}  {t:10.6f}  {r:5.2f}x")

---

### 4. 绘制 BER-SNR 曲线

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(snr_range, [max(b, 1e-6) for b in ber_list], "o-", label="BPSK sim")
theory = [0.5 * erfc(np.sqrt(10**(s/10))) for s in snr_range]
ax.semilogy(snr_range, [max(b, 1e-6) for b in theory], "s--", label="Theory Q(sqrt(2Eb/N0))")
ax.set_xlabel("Eb/N0 (dB)")
ax.set_ylabel("Bit Error Rate")
ax.set_title("Uncoded BPSK BER in AWGN")
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.5)
ax.set_ylim(bottom=1e-5)
plt.show()

---

## 课后实践

请补全下方 BPSK 手动调制与解调流程中的 **4 处空缺**（每处一行代码），在 SNR=6 dB 下对 1000 比特完成星座映射→RRC成型→AWGN→匹配滤波→判决的完整链路，并统计误码率。

要求：

1. **调制（2 处）**：补全 BPSK 星座映射和 RRC 脉冲成型
2. **解调（2 处）**：补全匹配滤波和下采样判决

完成后运行 `python bpsk_practice.py`。

In [ ]:
%%writefile bpsk_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.phy.psk import PSKModulator
from nearlink_sdr.phy.channel import ChannelModel

sps = 4
snr_db = 6.0
num_bits = 1000

rng = np.random.default_rng(42)
tx_bits = rng.integers(0, 2, num_bits)
mod = PSKModulator(mod_type="BPSK", sps=sps)
ch = ChannelModel(snr_db=snr_db)

# ========== 调制 ==========
# Step 1: BPSK 星座映射 —— 比特 0→+1, 比特 1→-1
symbols = ______________
# 上采样（插零）
upsampled = np.zeros(len(symbols) * sps, dtype=complex)
upsampled[::sps] = symbols
# Step 2: RRC 脉冲成型
tx_signal = ______________

# AWGN 信道
rx_signal = ch.apply_awgn(tx_signal, sps)

# ========== 解调（补全 2 处空缺）==========
# Step 3: 匹配滤波 —— 与相同 RRC 滤波器卷积
matched = ______________
# Step 4: 下采样 + 判决 —— 实部<0 判为 1，≥0 判为 0
rx_bits = ______________

n = min(len(tx_bits), len(rx_bits))
errors = int(np.sum(tx_bits[:n] != rx_bits[:n]))
ber = errors / n
print(f"SNR={snr_db:.0f} dB | {n} bits | {errors} errors | BER={ber:.4e}")

执行以下命令进行编译并验证结果：


In [ ]:
!python bpsk_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/02.03_answer.txt
